# 3-6 数据透视表pivot_table

`pivot_table()` 类似 Excel 的数据透视表：先按指定字段分组，再对数值进行汇总。可以逐步增加参数，观察结果如何变化。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备数据路径

本节读取本章 `data` 文件夹中的模拟练习数据，使用 `pathlib` 处理相对路径。

In [ ]:
from pathlib import Path

# 从本章目录运行时使用 data；从项目根目录运行时定位到本章。
data_dir = Path("data")
if not data_dir.is_dir():
    data_dir = Path("第3章-Pandas") / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("未找到本章的 data 文件夹。")

data_dir.as_posix()

In [ ]:
df = pd.read_excel(data_dir / "sales-funnel.xlsx", engine="openpyxl")
df.head()

## 2. 常用参数

| 参数 | 含义 |
|---|---|
| `index` | 结果的行分组字段 |
| `columns` | 结果的列分组字段 |
| `values` | 要统计的数值列 |
| `aggfunc` | 聚合方式，默认 `"mean"` |
| `fill_value` | 填充结果表中的缺失值 |
| `margins` | 是否增加总计 |
| `margins_name` | 总计标签名称 |

本例 `Price` 表示每条记录的模拟金额，`Quantity` 表示数量；按 `Price` 求和仅演示该字段的汇总。

## 3. 从简单透视表开始

### 3.1 按经理分组

明确写出 `values="Price"`，避免现代 Pandas 尝试对文本列求均值。

In [ ]:
df.pivot_table(index="Manager", values="Price", aggfunc="mean")

### 3.2 按经理、销售人员两层分组

`index` 可以是列表，结果行索引形成两层。

In [ ]:
df.pivot_table(index=["Manager", "Rep"], values="Price", aggfunc="mean")

## 4. 指定聚合方式

可以同时计算总和、有效数量、均值。字符串聚合名含义明确，也避免直接传入 `np.sum` 等函数时的版本兼容提示。

In [ ]:
df.pivot_table(index=["Manager", "Rep"], values="Price", aggfunc=["sum", "count", "mean"])

`len` 统计组内总行数；`count` 只统计非缺失值。本文件的 `Price` 没有缺失，两者结果一致；下面保留 `len` 的示例用于对比。

In [ ]:
df.pivot_table(index=["Manager", "Rep"], values="Price", aggfunc=len)

## 5. 将产品展开为列

`columns="Product"` 让每类产品成为一列，没有对应记录的位置显示缺失值。

In [ ]:
df.pivot_table(index=["Manager", "Rep"], columns="Product", values="Price", aggfunc="sum")

### 5.1 填充缺失值

`fill_value=0` 将透视结果中的空白填为 0；这不代表修改了原始数据。

In [ ]:
df.pivot_table(index=["Manager", "Rep"], columns="Product", values="Price", aggfunc="sum", fill_value=0)

### 5.2 添加总计

`margins=True` 添加行、列总计。总计采用相同的 `aggfunc`；当统计均值时，总计是从原数据重算均值，不能直接把各组均值相加。

In [ ]:
df.pivot_table(index=["Manager", "Rep"], columns="Product", values="Price", aggfunc="sum", fill_value=0, margins=True, margins_name="总计")

## 6. 不同列使用不同统计方式

`aggfunc` 传入字典，可以让金额取均值、数量求和。结果列索引包含统计字段与产品类别两层。

In [ ]:
df.pivot_table(
    index=["Manager", "Rep"], columns="Product",
    values=["Price", "Quantity"],
    aggfunc={"Price": "mean", "Quantity": "sum"}, fill_value=0
)

## 7. 要点总结

- 先确定行分组 `index`、数值 `values` 和统计方法 `aggfunc`。
- `columns` 把类别展开到列，多个分组字段会形成多层索引。
- `fill_value` 填充透视结果，`margins` 添加按同一规则计算的总计。
- `aggfunc` 可以是名称、函数、列表，也可以按字段指定字典。